In [ ]:
# =============================================================================
# jobs/35_build_warning_area_exposure.py
# Exposure of every flood warning area and flood alert area, counted from the address points, and what the areas miss
# (plan: docs/EXPOSURE_AND_RISK_GRID_PLAN.md). Restricted schema: built from address-level data.
#
# INPUTS  config.TBL_EA_FWA, config.TBL_EA_FAA   the warning and alert area polygons (job 03)
#         config.TBL_ADDRESS_POINTS              job 33: people, households, over-65s, deprivation, constituency and flood band at each address
#         config.TBL_CRITICAL_SITES              job 31 (optional): counted per area by group and band
# OUTPUT  config.TBL_AREA_EXPOSURE        one row per area (kind, area_id): details, then each measure four ways, see below, plus the share of its
#                                         people in the rivers and sea High or Medium band, in the most deprived fifth, and the critical sites in it
#         config.TBL_AREA_OVERLAPS        each pair of areas of one kind that share addresses (kind, area_a, area_b, addresses, people)
#         config.TBL_AREA_CONST_EXPOSURE  each area split by constituency (kind, area_id, pcon, addresses, people, households, over65, people_rs_high_medium)
#         config.TBL_AREA_COVERAGE        distinct people covered by any warning area, by any alert area, and in all: the gap is what they miss
#         config.TBL_CONST_EXPOSURE       per constituency: people by band, and how many people (and in High or Medium) are in a warning or alert area
#
# FOUR VIEWS OF EACH MEASURE (people, households, addresses, over-65s, people in the High or Medium band): areas overlap, so a person can be in
# several. gross = everyone in the area (for one area, not for adding up); unique = covered by this area alone (adds up across areas);
# split = each person shared equally among the areas covering them (adds up to the true total); people_shared = gross minus unique.
# Warnings are counted against warnings and alerts against alerts: a warning inside an alert does not make its people shared.
#
# COASTAL, FLUVIAL AND SURFACE WATER: the bands here are rivers and sea only. area_source says what the area is for (fluvial, coastal, tidal,
# groundwater), from its code. Groundwater areas have no rivers and sea band to compare: read them for people only. No surface water.
#
# WHAT TO CHECK (printed): areas read and any that are lines or empty (they cannot hold addresses and are listed); people covered against all;
# the gross-to-split ratio (how much a plain sum over areas double counts); the people in High or Medium that no warning area covers
# (the gap); the highest areas; whether the constituency codes match those the lookup table uses.
#
# Widgets: tiles, workers.
# =============================================================================

import os
import sys

_nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
if not _nb_path.startswith("/Workspace"):
    _nb_path = "/Workspace" + _nb_path
PROJECT_ROOT = os.path.dirname(os.path.dirname(_nb_path))
sys.path.insert(0, PROJECT_ROOT)

# these are not on every cluster; installed if missing, and Python restarted only if they still cannot be imported
def _have():
    try:
        import pyproj  # noqa: F401
        import shapely  # noqa: F401
        return True
    except ImportError:
        return False


if not _have():
    get_ipython().run_line_magic("pip", "install -q pyproj shapely")
    import importlib
    importlib.invalidate_caches()
    if not _have():
        print("Packages installed but not importable in this session: restarting Python. Run all again from the top.")
        dbutils.library.restartPython()

import importlib
import threading
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from pyspark.sql import functions as F

import config
import importlib as _il

_il.reload(config)   # a Python session that ran an earlier version of config.py would otherwise miss tables added since
import utils.area_exposure as _ae
importlib.reload(_ae)
from utils.area_exposure import (KINDS, SOURCE_BY_CODE, area_cell_scores, UNIQUE_SPLIT, combine_tiles, constituency_with_gap, coverage_with_gap, finish_area_geographies, finish_areas, checkpoint_from_bytes, checkpoint_to_bytes, site_membership, site_stats, tile_signature, tile_stats, to_bng_polygons)
from utils.helpers import get_spark

spark = get_spark()


def show(df):
    display(df.fillna("").astype(str) if len(df) else df)


dbutils.widgets.text("tiles", "", "Only these tiles, for a trial (comma separated). Blank is all. A trial does not write the tables")
dbutils.widgets.text("workers", "1", "Tiles to run at once")
dbutils.widgets.dropdown("reuse_tiles", "no", ["no", "yes"], "Reuse the saved tile results (skips the long tile step if they match)")
dbutils.widgets.text("checkpoint_table", config.TBL_JOB35_CHECKPOINT, "Restricted table the tile results are saved in (survives a cluster restart)")
ONLY = [t.strip().upper() for t in dbutils.widgets.get("tiles").split(",") if t.strip()]
WORKERS = max(1, int(dbutils.widgets.get("workers") or 1))
REUSE = dbutils.widgets.get("reuse_tiles") == "yes"
CKPT = dbutils.widgets.get("checkpoint_table").strip() or config.TBL_JOB35_CHECKPOINT
print(f"Workers {WORKERS}. Trial tiles: {ONLY or 'none (all)'}.")


In [ ]:
# =============================================================================
# 1. THE AREAS: polygons in British National Grid, with what each is for
# =============================================================================

frames = []
for table, kind in ((config.TBL_EA_FWA, "flood_warning"), (config.TBL_EA_FAA, "flood_alert")):
    t = spark.table(table).toPandas()
    t["kind"] = kind
    frames.append(t)
raw = pd.concat(frames, ignore_index=True)
raw = raw.rename(columns={"ea_area_code": "area_id", "ea_area_name": "area_name"})
dups = raw.duplicated(["kind", "area_id"]).sum()
if dups:
    print(f"{dups} repeated (kind, area code) rows dropped, first kept: a repeat would be counted twice.")
    raw = raw.drop_duplicates(["kind", "area_id"]).reset_index(drop=True)

geoms, status = to_bng_polygons(raw["geometry"].tolist())
raw["geom_status"] = status
print("Areas read, by kind and geometry:")
show(raw.groupby(["kind", "geom_status"]).size().rename("areas").reset_index())
bad = raw[raw["geom_status"] != "ok"]
if len(bad):
    print("Areas that cannot hold addresses (lines, points or empty) and are left out of the counts:")
    show(bad[["kind", "area_id", "area_name", "type_code", "geom_status"]].head(30))

ok = raw["geom_status"] == "ok"
areas = raw[ok].reset_index(drop=True)
polys = [g for g, s in zip(geoms, status) if s == "ok"]
areas["area_km2"] = np.round([g.area / 1e6 for g in polys], 3)
areas["area_source"] = [SOURCE_BY_CODE.get(str(c)[2:3], "unknown") if len(str(c)) >= 3 else "unknown" for c in areas["type_code"]]
meta = areas[["kind", "area_id", "area_name", "type_code", "type_text", "area_source", "river_or_sea", "county", "parent", "area_km2"]].copy()
print("\nAreas by kind and what they are for:")
show(meta.groupby(["kind", "area_source"]).size().rename("areas").reset_index())


In [ ]:
# =============================================================================
# 2. EACH TILE OF ADDRESS POINTS AGAINST ALL AREAS
# Tiles of more than 400,000 points (cities) run one at a time after the small ones, as in job 33.
# =============================================================================

POINT_COLS = ["easting", "northing", "people", "households", "over65", "imd_decile", "point_source", "pcon", "lad", "ctyua", "ward", "pfa", "rgn", "oa",
              "rofrs_rank", "fz_rank", "fz_sea_rank", "fz_river_rank", "fz_river_and_sea_rank"]
points_t = spark.table(config.TBL_ADDRESS_POINTS)
tile_counts = points_t.groupBy("tile").count().toPandas().dropna().sort_values("tile")
tile_counts = tile_counts[tile_counts["tile"] != ""]
if ONLY:
    tile_counts = tile_counts[tile_counts["tile"].isin(ONLY)]
    if tile_counts.empty:
        raise ValueError(f"None of the tiles {ONLY} hold points. Tiles are named like SE00, SE05, SE50, SE55.")
print(f"{len(tile_counts)} tiles; the biggest has {int(tile_counts['count'].max()):,} points.")

# fire and rescue authority (from the output area) and combined authority (from the district) are not on the address points: job 32 writes the lookups
EXTRA_MAPS = {}
if spark.catalog.tableExists(config.TBL_OA_FIRE):
    _f = spark.table(config.TBL_OA_FIRE).toPandas()
    EXTRA_MAPS["fire_and_rescue_authority"] = ("oa", dict(zip(_f["oa"].astype(str), _f["code"].astype(str))))
if spark.catalog.tableExists(config.TBL_DISTRICT_LOOKUP):
    _d = spark.table(config.TBL_DISTRICT_LOOKUP).toPandas()
    for _g in sorted(_d["geography"].unique()):
        if _g == "fire_and_rescue_authority" and "fire_and_rescue_authority" in EXTRA_MAPS:
            continue                                  # the boundary based assignment is the better one
        _x = _d[_d["geography"] == _g].drop_duplicates("district_code")
        EXTRA_MAPS[_g] = ("lad", dict(zip(_x["district_code"].astype(str), _x["code"].astype(str))))
print(f"Responder lookups from job 32: {sorted(EXTRA_MAPS) or 'none (run job 32 first for fire and combined authorities)'}.")

results, log = [], []
lock = threading.Lock()
AREA_IDS, AREA_KINDS = areas["area_id"].tolist(), areas["kind"].tolist()


def do_tile(tile, n_points, i, n):
    t0 = time.time()
    try:
        p = points_t.filter(F.col("tile") == tile).select(*POINT_COLS).toPandas()
        s = tile_stats(p, polys, AREA_IDS, AREA_KINDS, extra_maps=EXTRA_MAPS)
        with lock:
            results.append(s)
            log.append({"tile": tile, "status": "done", "points": len(p), "seconds": round(time.time() - t0, 1)})
        print(f"[{i}/{n}] {tile}: {len(p):,} points, {time.time() - t0:.0f}s")
        del p
    except Exception as e:
        with lock:
            log.append({"tile": tile, "status": "FAILED: " + str(e)[:150]})
        print(f"[{i}/{n}] {tile}: FAILED {str(e)[:150]}")


SIGNATURE = {**tile_signature(areas, tile_counts), "extra": sorted(EXTRA_MAPS)}
LOADED = False
if REUSE and not ONLY:
    if spark.catalog.tableExists(CKPT):
        _row = spark.table(CKPT).orderBy(F.col("saved_at").desc()).limit(1).collect()[0]
        _res, _log, _why = checkpoint_from_bytes(_row["payload"], SIGNATURE)
        _saved_at = _row["saved_at"]
    else:
        _res, _log, _why, _saved_at = None, None, "no saved results in that table", None
    if _res is not None:
        results, log, LOADED = _res, _log, True
        print(f"Reusing the tile results saved at {_saved_at} in {CKPT}: {len(log)} tiles. The tile step is skipped.")
    else:
        print(f"Not reusing saved results: {_why}. Running the tiles.")
if not LOADED:
    LARGE = 400_000
    rows = [(r.tile, int(r["count"]), i, len(tile_counts)) for i, (_, r) in enumerate(tile_counts.iterrows(), 1)]
    small = [w for w in rows if w[1] <= LARGE]
    large = [w for w in rows if w[1] > LARGE]
    print(f"{len(small)} tiles up to {LARGE:,} points (run {WORKERS} at once) and {len(large)} larger ones (run one at a time).")
    if WORKERS == 1:
        for w in small:
            do_tile(*w)
    else:
        with ThreadPoolExecutor(max_workers=WORKERS) as pool:
            for f in as_completed([pool.submit(do_tile, *w) for w in small]):
                f.result()
    for w in large:
        do_tile(*w)

failed = [r for r in log if r["status"].startswith("FAILED")]
print(f"\nTiles done: {sum(1 for r_ in log if r_['status'] == 'done')}. Failed: {len(failed)}.")
if failed:
    show(pd.DataFrame(failed))

if not LOADED and not failed and not ONLY and results:
    # the per-tile sums, added up once, saved in a restricted table so a restart or an overnight disconnect does not repeat the tile step
    _combined = {k: combine_tiles([r[k] for r in results], None) for k in results[0]}
    _blob = checkpoint_to_bytes([_combined], log, SIGNATURE)
    _pdf = pd.DataFrame({"saved_at": [pd.Timestamp.now(tz="UTC").isoformat()], "n_tiles": [len(log)], "payload": [_blob]})
    spark.createDataFrame(_pdf).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(CKPT)
    spark.sql("COMMENT ON TABLE " + CKPT + " IS 'The per-tile sums job 35 saves after its tile step so a restart does not repeat it: area, pair, constituency, responder and coverage sums, with the areas and tile counts they were made from. One row, overwritten. Restricted. Job 35.'")
    print(f"Tile results saved in {CKPT} ({len(_blob) / 1e6:.1f} MB). After a restart, set reuse_tiles to yes to skip the tile step.")


In [ ]:
# =============================================================================
# 3. ADD THE TILES UP, ADD THE CRITICAL SITES, WORK OUT THE DERIVED COLUMNS, WRITE
# A trial (tiles set) shows the result but does not write the tables.
# =============================================================================

area_sum = combine_tiles([r["area"] for r in results], None)
pairs_sum = combine_tiles([r["pairs"] for r in results], None)
ap_sum = combine_tiles([r["area_pcon"] for r in results], None)
cov_sum = combine_tiles([r["coverage"] for r in results], None)
pcon_sum = combine_tiles([r["pcon"] for r in results], None)
geo_sum = combine_tiles([r["area_geo"] for r in results], None)

exposure = finish_areas(area_sum, meta)

# critical sites per area, if job 31 has run
sites_cols = ["sites", "sites_health", "sites_education", "sites_emergency", "sites_utilities", "sites_rs_high_medium", "sites_fz3"]
if spark.catalog.tableExists(config.TBL_CRITICAL_SITES):
    sites = spark.table(config.TBL_CRITICAL_SITES).select("site_id", "name", "site_type", "easting", "northing", "site_group", "rofrs_rank", "fz_rank").toPandas()
    st = site_stats(sites, polys, areas["area_id"].tolist(), areas["kind"].tolist()).reset_index()
    exposure = exposure.merge(st, on=["kind", "area_id"], how="left")
    area_sites = site_membership(sites, polys, areas["area_id"].tolist(), areas["kind"].tolist())
    print(f"Critical sites: {len(sites):,}; {int(st['sites'].sum()):,} area memberships.")
else:
    area_sites = None
    print("No critical sites table (job 31 has not run): the site columns are zero.")
for c in sites_cols:
    exposure[c] = exposure[c].fillna(0.0) if c in exposure else 0.0

# a risk score per area from the 100 m grid (job 34), if it has been built: the people-weighted mean of the cell scores inside the area
if spark.catalog.tableExists(config.TBL_EXPOSURE_GRID):
    grid = spark.table(config.TBL_EXPOSURE_GRID).select("easting", "northing", "people", "risk_score").toPandas()
    sc = area_cell_scores(grid, polys, areas["area_id"].tolist(), areas["kind"].tolist()).reset_index()
    exposure = exposure.merge(sc, on=["kind", "area_id"], how="left")
    print(f"Area risk score from {len(grid):,} grid cells: {int(sc['risk_score'].notna().sum()):,} of {len(exposure):,} areas scored; median {sc['risk_score'].median():.1f}, highest {sc['risk_score'].max():.1f}.")
    del grid
else:
    for c in ("grid_cells", "risk_score", "risk_score_max"):
        exposure[c] = np.nan
    print("No grid table (job 34 has not run): the area risk score is blank.")

overlaps = pairs_sum.reset_index().rename(columns={"area_id_a": "area_a", "area_id_b": "area_b"}) if len(pairs_sum) else pd.DataFrame(columns=["kind", "area_a", "area_b", "addresses", "people"])
area_const = ap_sum.reset_index().rename(columns={"pcon": "constituency_code"})
names = spark.table(config.TBL_GEOGRAPHY_NAMES).toPandas() if spark.catalog.tableExists(config.TBL_GEOGRAPHY_NAMES) else None
area_geo = finish_area_geographies(geo_sum, exposure, names)
print(f"{len(area_geo):,} area by responder rows" + ("" if names is not None else " (no names table: job 32 has not run, so names are blank)") + ". Unnamed codes per geography:")
print(area_geo.assign(unnamed=area_geo["name"].isna()).groupby("geography")["unnamed"].sum().to_string())
coverage = coverage_with_gap(cov_sum)
const = constituency_with_gap(pcon_sum)

# critical sites per constituency (England), from the constituency boundaries: counts by group and band on the constituency table, and the list of sites
const_sites = None
site_cols = ["sites", "sites_health", "sites_education", "sites_emergency", "sites_utilities", "sites_rs_high_medium"]
if area_sites is not None and spark.catalog.tableExists(config.TBL_CONSTITUENCIES):
    cb = spark.table(config.TBL_CONSTITUENCIES).select("constituency_id", "geometry").toPandas()
    cb = cb[cb["constituency_id"].astype(str).str.startswith("E") & cb["geometry"].notna()].reset_index(drop=True)
    cgeoms, cstatus = to_bng_polygons(cb["geometry"].tolist())
    ok = [i for i, s_ in enumerate(cstatus) if s_ == "ok"]
    cids = [cb["constituency_id"].iloc[i] for i in ok]
    cpolys = [cgeoms[i] for i in ok]
    print(f"Constituency boundaries: {len(cids)} of {len(cb)} usable.")
    ck = ["constituency"] * len(cids)
    cs = site_stats(sites, cpolys, cids, ck).reset_index().rename(columns={"area_id": "constituency_code"}).drop(columns=["kind", "sites_fz3"])
    const = const.merge(cs, on="constituency_code", how="left")
    const_sites = site_membership(sites, cpolys, cids, ck).rename(columns={"area_id": "constituency_code"}).drop(columns=["kind"])
    print(f"Critical sites by constituency: {int(cs['sites'].sum()):,} memberships; {int((cs['sites'] > 0).sum())} of {len(cids)} constituencies hold at least one.")
for c_ in site_cols:
    const[c_] = const[c_].fillna(0.0) if c_ in const else 0.0
print(f"{len(exposure):,} areas, {len(overlaps):,} overlapping pairs, {len(area_const):,} area by constituency rows, {len(const):,} constituency codes.")


def clean(pdf):
    out = pdf.copy()
    for c in out.columns:
        if out[c].dtype == object:
            out[c] = out[c].astype("object").where(out[c].notna(), None)
            out[c] = out[c].map(lambda v: None if v is None else str(v))
    return out


def write(pdf, table, comment):
    spark.createDataFrame(clean(pdf)).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table)
    spark.sql("COMMENT ON TABLE " + table + " IS '" + comment.replace("'", "''") + "'")
    print(f"Written {len(pdf):,} rows to {table}.")


if failed:
    print("Not writing: some tiles failed. Re-run them first.")
elif ONLY:
    print("Trial run: the tables are not written.")
else:
    write(exposure, config.TBL_AREA_EXPOSURE, "Exposure of each EA flood warning and flood alert area, counted from the address points (Census 2021 shares, IoD2025, rivers and sea band). Each measure as gross, unique (this area alone) and split (shared equally between overlapping areas of the same kind). Rivers and sea only, no surface water; area_source says fluvial, coastal, tidal or groundwater. Restricted. Job 35.")
    write(overlaps, config.TBL_AREA_OVERLAPS, "Pairs of EA flood warning (or alert) areas that share address points, with the addresses and people they share. Job 35.")
    write(area_const, config.TBL_AREA_CONST_EXPOSURE, "Each EA flood warning or alert area split by parliamentary constituency (NSUL PCON24CD): addresses, people, households, over-65s, people in the rivers and sea High or Medium band and in Flood Zone 3. Job 35.")
    write(coverage, config.TBL_AREA_COVERAGE, "Distinct people covered by any EA flood warning area, by any flood alert area, and in all of England, so the gap (people in the High or Medium band that no area covers) is visible. Job 35.")
    if area_sites is not None:
        write(area_sites, config.TBL_AREA_SITES, "Each critical site inside each EA flood warning or alert area, with its rivers and sea band (worst first). A site in two areas has a row for each. Restricted. Job 35.")
    write(area_geo, config.TBL_AREA_GEOGRAPHIES, "Each EA flood warning or alert area split by the geographies responders work to (local authority district, county or unitary, ward, constituency, police force area, region): addresses, people, households, over-65s, people in the rivers and sea High or Medium band, the share of the area people, and the name where job 32 has it. Fire and rescue and combined authorities are not here yet. Restricted. Job 35.")
    if const_sites is not None:
        write(const_sites, config.TBL_CONST_SITES, "Each critical site inside each England parliamentary constituency, with its rivers and sea band (worst first). Restricted. Job 35.")
    write(const, config.TBL_CONST_EXPOSURE, "Per parliamentary constituency: people by rivers and sea band, in Flood Zone 3, and how many are in a flood warning or alert area. Job 35.")


In [ ]:
# =============================================================================
# 4. CHECKS
# =============================================================================

print("Coverage: distinct people covered by any area of a kind, against everyone in the points (the gap is what the areas miss):")
show(coverage[["kind", "addresses", "people", "people_rs_high_medium", "people_not_in_area", "people_high_medium_not_in_area"]].round(0))

for k in ("flood_warning", "flood_alert"):
    e = exposure[exposure["kind"] == k]
    gross, split = e["people"].sum(), e["people_split"].sum()
    true_total = float(coverage.loc[coverage["kind"] == k, "people"].iloc[0])
    print(f"\n{k}: {len(e):,} areas. Summing gross people over areas gives {gross:,.0f}; the distinct total is {true_total:,.0f} (split adds up to {split:,.0f}, which must equal the distinct total). "
          f"A plain sum overstates by {gross / true_total - 1:.0%}." if true_total else f"\n{k}: no people covered.")
    print(f"   areas with no addresses at all: {int((e['addresses'] == 0).sum()):,} of {len(e):,}")

print("\nWarning areas by what they are for (rivers and sea bands do not apply to groundwater):")
w = exposure[exposure["kind"] == "flood_warning"]
show(w.groupby("area_source").agg(areas=("area_id", "size"), people=("people", "sum"), people_unique=("people_unique", "sum"), people_split=("people_split", "sum"),
                                   high_medium_split=("people_rs_high_medium_split", "sum")).round(0).reset_index())

print("\nThe 15 warning areas with the most people in the High or Medium band (split, so no person is counted twice):")
show(w.sort_values("people_rs_high_medium_split", ascending=False).head(15)[["area_id", "area_name", "area_source", "county", "people", "people_unique", "people_split",
                                                                               "people_rs_high_medium", "share_people_high_medium", "imd_mean_decile", "sites"]].round(2))
print("\nThe 10 pairs of warning areas sharing the most people:")
show(overlaps[overlaps["kind"] == "flood_warning"].sort_values("people", ascending=False).head(10).round(0))

lookup_ids = set(spark.table(config.TBL_EA_CONST_LOOKUP).select("constituency_id").distinct().toPandas()["constituency_id"].astype(str))
codes = set(const["constituency_code"].astype(str)) - {""}
print(f"\nConstituency codes: {len(codes)} in the address points, {len(lookup_ids)} in the lookup table, {len(codes & lookup_ids)} in both. "
      "If few match, the two use different identifiers and the app needs a mapping.")


In [ ]:
# Exit is in its own cell: in an interactive run Databricks replaces a cell's output with the exit message,
# so the checks above would be hidden if this line shared their cell.
dbutils.notebook.exit("success" if not failed else "finished with failed tiles")
